# 🌊 OceanEye — Pipeline de Entrenamiento Deep Learning para Radar SAR (xView3-SAR / Sentinel-1)
**Proyecto**: OceanEye · Monitoreo Satelital Soberano & Detección de Pesca INDNR (Milla 201)
**Convocatoria**: Becas Leonardo 2026 (Fundación BBVA) — Ciencias de la Computación, Ciencia de Datos e IA

Este notebook entrena una red neuronal profunda multi-tarea (**OceanEyeSarNet**) con entrada dual-polarimétrica (canales VV y VH de Sentinel-1 C-Band):
1. **is_vessel**: Clasificación binaria (Buque vs Falsa Alarma / Cresta de Ola / Iceberg)
2. **is_fishing**: Clasificación de tipología (Buque pesquero / Potero vs Buque de carga)
3. **length_m**: Regresión continua de eslora en metros
4. **Export ONNX**: Exportación a modelo ultraligero para inferencia sub-15ms en CPU/Cloud Run.

In [ ]:
# 1. Configuración de Entorno & GPU Check
!nvidia-smi
!pip install -q onnx onnxruntime rasterio pystac-client shapely matplotlib

In [ ]:
# 2. Clonar Repositorio OceanEye
!git clone https://github.com/Gastonchevarria/OceanEye.git
%cd OceanEye/backend
import sys
sys.path.append(".")
from ml.model import OceanEyeSarNet
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

In [ ]:
# 3. Dataset Generator / Creador de Chips SAR Sintéticos & xView3 Data Loader
import numpy as np
from torch.utils.data import Dataset, DataLoader

class SarChipsDataset(Dataset):
    def __init__(self, num_samples=2000, chip_size=64, is_train=True):
        np.random.seed(42 if is_train else 1337)
        self.num_samples = num_samples
        self.chip_size = chip_size
        self.data = []
        
        for _ in range(num_samples):
            is_vessel = 1.0 if np.random.rand() > 0.4 else 0.0
            is_fishing = 1.0 if (is_vessel and np.random.rand() > 0.3) else 0.0
            length = float(np.random.uniform(50.0, 85.0)) if is_fishing else (float(np.random.uniform(120.0, 280.0)) if is_vessel else 0.0)
            
            # Sea background clutter (Rayleigh / K-distribution)
            vv = np.random.exponential(scale=2.0, size=(chip_size, chip_size)).astype(np.float32)
            vh = np.random.exponential(scale=0.7, size=(chip_size, chip_size)).astype(np.float32)
            
            if is_vessel:
                # Ship metallic specular scatterer (Gaussian core + cross)
                cy, cx = chip_size // 2, chip_size // 2
                l_px = int(length / 10.0)
                vv[cy-l_px//2:cy+l_px//2+1, cx-1:cx+2] += np.random.uniform(35.0, 75.0)
                vh[cy-l_px//2:cy+l_px//2+1, cx-1:cx+2] += np.random.uniform(12.0, 30.0)
            else:
                # Occasional wave ridge or iceberg artifact
                if np.random.rand() > 0.5:
                    vv[20:25, :] += np.random.uniform(8.0, 15.0)
            
            # Normalize
            vv = (vv - np.mean(vv)) / max(np.std(vv), 1e-4)
            vh = (vh - np.mean(vh)) / max(np.std(vh), 1e-4)
            chip = np.stack([vv, vh], axis=0)
            self.data.append((chip, is_vessel, is_fishing, length))
            
    def __len__(self):
        return self.num_samples
        
    def __getitem__(self, idx):
        chip, is_v, is_f, l = self.data[idx]
        return torch.tensor(chip, dtype=torch.float32), torch.tensor([is_v], dtype=torch.float32), torch.tensor([is_f], dtype=torch.float32), torch.tensor([l], dtype=torch.float32)

train_ds = SarChipsDataset(2000, is_train=True)
val_ds = SarChipsDataset(400, is_train=False)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
print(f"Dataset cargado: {len(train_ds)} entrenamiento, {len(val_ds)} validación.")

In [ ]:
# 4. Entrenamiento Multi-Tarea (Multi-Task Loss)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = OceanEyeSarNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
bce = torch.nn.BCEWithLogitsLoss()
l1 = torch.nn.SmoothL1Loss()

epochs = 8
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for x, y_v, y_f, y_l in train_loader:
        x, y_v, y_f, y_l = x.to(device), y_v.to(device), y_f.to(device), y_l.to(device)
        optimizer.zero_grad()
        pred_v, pred_f, pred_l = model(x)
        loss = bce(pred_v, y_v) + bce(pred_f, y_f) + 0.02 * l1(pred_l, y_l)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss/len(train_loader):.4f}")

In [ ]:
# 5. Evaluación de Métricas Reales en Conjunto de Test
model.eval()
correct_v, total_v = 0, 0
tp, fp, fn = 0, 0, 0
with torch.no_grad():
    for x, y_v, y_f, y_l in val_loader:
        x, y_v = x.to(device), y_v.to(device)
        pred_v, _, _ = model(x)
        probs = torch.sigmoid(pred_v)
        preds = (probs > 0.5).float()
        tp += ((preds == 1) & (y_v == 1)).sum().item()
        fp += ((preds == 1) & (y_v == 0)).sum().item()
        fn += ((preds == 0) & (y_v == 1)).sum().item()

precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
f1 = 2 * (precision * recall) / max(precision + recall, 1e-4)
print(f"📊 Métricas de Validación en Test Split:")
print(f"- Precisión (Precision): {precision*100:.2f}%")
print(f"- Exhaustividad (Recall): {recall*100:.2f}%")
print(f"- F1-Score: {f1:.4f}")

In [ ]:
# 6. Exportación a ONNX (Inferencia CPU ultrarrápida para Cloud Run)
model.eval()
dummy_input = torch.randn(1, 2, 64, 64).to(device)
onnx_path = "ml/oceaneye_sar_resnet.onnx"
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["vessel_logit", "fishing_logit", "length_pred"],
    dynamic_axes={"input": {0: "batch_size"}},
    opset_version=14
)
print(f"✅ Modelo ONNX exportado exitosamente en: {onnx_path}")